# OASIS-2 clinical validation (read-only, CPU)

Run this notebook in an **authorized private Google Colab session**, from top to
bottom. It reviews existing clinical linkage metadata, samples existing Analyze
pairs, and proposes a subject-level split **in memory only**. It never downloads
medical data, edits source files, creates diagnoses, trains a model, or changes
Task C's model, preprocessing, calibration or JSON contract.

**No final clinical diagnoses or Task A/C labels are confirmed.** Nondemented +
CDR 0 is a cognitive comparison candidate, not proof of neurologically normal MRI.
Demented does not mean confirmed Alzheimer's; CDR 0.5 does not automatically mean
MCI. Converted subjects retain their original per-visit Group/CDR values.

Colab may save outputs and configuration into the notebook. Keep the working
copy private; clear all outputs and restore placeholder paths before sharing or
committing it. Never display records, identifiers, filenames, clinical Excel rows,
raw exceptions, image slices or in-memory split assignments. This repository
contains only this unexecuted template and synthetic tests; Codex has no private
Drive access. Use a CPU runtime; a GPU is unnecessary.

## 1. Public code and reproducibility

The clinical policy and readiness modules must already have been reviewed. The
setup below installs **public Python packages and public project code only** into
Colab's ephemeral `/content` workspace. It neither fetches a dataset nor touches
PR #2. It checks out the exact reviewed module baseline and refuses to reuse an
incompatible or modified source checkout. Package installation requires network;
subsequent audits read mounted files only. Do not run setup against a checkout
containing private data. Use Python 3.12+ for the pinned packages below and
restart the runtime after changing dependency versions.

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path("/content/BrainAgenticthon-2026-clinical-validation")
MODULE_COMMIT = "8083fa96e4e503745a56a9e4ec7f560915c10468"
PUBLIC_REPO = "https://github.com/abdulrahmantaani24-dotcom/BrainAgenticthon-2026.git"
PUBLIC_PACKAGES = ("numpy==2.5.4", "nibabel==5.4.2")
INSTALL_PUBLIC_DEPENDENCIES = True

In [ ]:
import subprocess


def public_setup():
    if sys.version_info < (3, 12):
        raise RuntimeError("These pinned packages require a Python 3.12+ Colab runtime.")
    if INSTALL_PUBLIC_DEPENDENCIES:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *PUBLIC_PACKAGES],
            check=True, capture_output=True,
        )
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--quiet", "--no-checkout", PUBLIC_REPO, str(REPO_DIR)],
            check=True, capture_output=True,
        )
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "checkout", "--quiet", "--detach", MODULE_COMMIT],
            check=True, capture_output=True,
        )
    revision = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True,
    ).strip()
    if revision != MODULE_COMMIT:
        raise RuntimeError("Use a fresh Colab public-code directory at the reviewed commit.")
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "diff", "--quiet", "HEAD", "--", "src"],
        check=True, capture_output=True,
    )
    source = str(REPO_DIR / "src")
    if source not in sys.path:
        sys.path.insert(0, source)


try:
    public_setup()
except Exception:
    raise RuntimeError("Public setup failed; check runtime and reviewed code setup.") from None

In [ ]:
import hashlib
import io
import json
import logging
import math
import random
import warnings
from collections import Counter, defaultdict
from contextlib import redirect_stderr, redirect_stdout
from importlib.metadata import version

import nibabel as nib
import numpy as np

from brainagenticthon.preprocessing.data_validator import NIfTIValidator
from brainagenticthon.preprocessing.manifest_schema import ManifestCollection, ManifestRecord
from brainagenticthon.preprocessing.oasis2_clinical_policy import (
    CDR_VALUES,
    GROUPS,
    LinkageAttestation,
    audit_clinical_manifest,
    load_clinical_images,
)
from brainagenticthon.preprocessing.split_manager import SplitManager
from brainagenticthon.preprocessing.task_a_readiness import audit_readiness

if np.__version__ != version("numpy") or nib.__version__ != version("nibabel"):
    raise RuntimeError("Restart the Colab runtime: loaded and installed package versions differ.")
if tuple(f"{name}=={version(name)}" for name in ("numpy", "nibabel")) != PUBLIC_PACKAGES:
    raise RuntimeError("Install the reviewed package pins or review updated versions explicitly.")

## 2. Private access and path configuration

Only mount Drive if your OASIS-2 access terms and institutional authorization
permit it. Edit **placeholder paths in your private working copy**, then set the
access acknowledgment. No credentials belong here. Raw images stay in a separate
source directory. The manifest is the existing JSON **array** with `patient_id`,
`visit_id`, `clinical_group`, `cdr`, `relative_path`, `dataset`, `diagnosis_source`,
null `task_a_label`/`task_c_label`, and `label_status`. By default no MMSE, split or
linkage status field is assumed. The parser distinguishes absent MMSE from null
MMSE; the two missing MMSE values in the original clinical source cannot be
identified from an export that omits the entire column.

Use an optional field map only for a separately documented schema. A separate
linkage attestation is optional: if omitted, the policy reports linkage as
unverified, rather than inventing record-level match status. Its fields are
`evidence_reference`, `matched_image_records`, `matched_subject_visit_keys`,
`unmatched_image_records`, and `ambiguous_image_records`. The attestation must
already exist; this notebook does not create or overwrite it. Prior observed
counts are **not** automatically supplied as an attestation.

In [ ]:
AUTHORIZED_PRIVATE_ACCESS = False  # Change only in your authorized private Colab copy.
DRIVE_MOUNT = Path("/content/drive")
PRIVATE_ROOT = DRIVE_MOUNT / "MyDrive" / "PRIVATE"
RAW_ROOT_RELATIVE = "OASIS-2/raw"
MANIFEST_RELATIVE = "verified_clinical/oasis2_clinical_manifest.json"
ATTESTATION_RELATIVE = None  # Or "reviews/oasis2-linkage-attestation.json".
FIELD_MAP = None  # Default exact private export profile; no guessed field names.
RECORDS_KEY = "records"  # Used only for an explicit object envelope, not a JSON array.
SAMPLE_SIZE = 3  # Unique Analyze pairs, not 3 image rows of the same pair.
SAMPLE_SEED = 42
MAX_SAMPLE_VOXELS = 32_000_000
MAX_PAIR_BYTES = 256 * 1024 * 1024
SPLIT_SEED = 42
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15

In [ ]:
def private_call(operation, *args, **kwargs):
    """Suppress library output and error details that could contain private paths."""
    previous = logging.root.manager.disable
    try:
        logging.disable(logging.CRITICAL)
        with (redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()),
              warnings.catch_warnings()):
            warnings.simplefilter("ignore")
            return operation(*args, **kwargs)
    except Exception:
        raise RuntimeError(
            "Private validation stopped. Check inputs privately; no records were changed."
        ) from None
    finally:
        logging.disable(previous)


def hidden_metadata(path):
    return any(part.startswith(".") or part == "__MACOSX" for part in Path(path).parts)


def private_path(root, relative):
    """Reject hidden metadata, traversal, absolute paths and symlink escapes."""
    if not isinstance(relative, str) or not relative or "\\" in relative or ":" in relative:
        raise ValueError("unsafe_relative_path")
    candidate = Path(relative)
    if candidate.is_absolute() or hidden_metadata(candidate):
        raise ValueError("unsafe_relative_path")
    root = Path(root).resolve()
    resolved = (root / candidate).resolve()
    if not resolved.is_relative_to(root):
        raise ValueError("path_outside_private_root")
    return resolved


def digest_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_private_inputs(root, raw_relative, manifest_relative, attestation_relative=None,
                        *, field_map=None, records_key="records"):
    raw_root = private_path(root, raw_relative)
    manifest_path = private_path(root, manifest_relative)
    if not raw_root.is_dir() or not manifest_path.is_file():
        raise ValueError("required_private_input_missing")
    images, manifest_digest = load_clinical_images(
        manifest_path, field_map=field_map, records_key=records_key,
    )
    if not images:
        raise ValueError("empty_manifest")
    fingerprints = {manifest_path: manifest_digest}
    linkage = None
    if attestation_relative is not None:
        path = private_path(root, attestation_relative)
        content = path.read_bytes()
        linkage = LinkageAttestation(**json.loads(content))
        fingerprints[path] = hashlib.sha256(content).hexdigest()
    return images, linkage, raw_root, fingerprints


def inputs_unchanged(fingerprints):
    return all(digest_file(path) == digest for path, digest in fingerprints.items())

In [ ]:
if not AUTHORIZED_PRIVATE_ACCESS:
    raise RuntimeError("Set the access acknowledgment only in an authorized private Colab copy.")
try:
    from google.colab import drive
except ImportError:
    raise RuntimeError("Drive mounting is available only in the intended Colab runtime.") from None

# Mounting requires Google's interactive consent; do not enter credentials into cells.
drive.mount(str(DRIVE_MOUNT))
images, linkage, raw_root, input_fingerprints = private_call(
    load_private_inputs, PRIVATE_ROOT, RAW_ROOT_RELATIVE, MANIFEST_RELATIVE,
    ATTESTATION_RELATIVE, field_map=FIELD_MAP, records_key=RECORDS_KEY,
)
print(json.dumps({"private_inputs_loaded": True, "attestation_supplied": linkage is not None}))

## 3. Read-only clinical policy and observed-count comparison

The existing `audit_clinical_manifest` runs without diagnosis evidence. It reuses
`task_a_readiness` and `SplitManager`, preserves original values and returns
aggregate warnings. The supplied observations below are **reference values**,
not assertions that this notebook has already audited private data. Differences
require review. No reference count fills missing fields or creates a label.

In [ ]:
OBSERVED_COUNTS = {
    "subjects": 150, "visits": 373, "images": 1368,
    "group_visits": {"Nondemented": 190, "Demented": 146, "Converted": 37},
    "group_subjects": {"Demented": 64, "Converted": 14},
    "group_cdr_visits": {
        "Nondemented": {"0.0": 188, "0.5": 2},
        "Demented": {"0.5": 102, "1.0": 41, "2.0": 3},
        "Converted": {"0.0": 18, "0.5": 19},
    },
    "cognitive_candidate_visits": 188, "cognitive_candidate_subjects": 72,
}


def compare_observed(report):
    counts = report["counts"]
    actual = {
        "subjects": counts["unique_subjects"],
        "visits": counts["unique_subject_visit_keys"],
        "images": counts["image_records"],
        "group_visits": counts["group_sessions"],
        "group_subjects": {g: counts["group_subjects"].get(g, 0)
                           for g in OBSERVED_COUNTS["group_subjects"]},
        "group_cdr_visits": counts["group_cdr_sessions"],
        "cognitive_candidate_visits": counts["cognitive_comparison_candidate_sessions"],
        "cognitive_candidate_subjects": counts["cognitive_comparison_candidate_subjects"],
    }
    return {
        "reference_scope": "user-supplied observations; not independently confirmed diagnoses",
        "comparisons": {key: {"observed_reference": expected, "actual": actual[key],
                              "matches": actual[key] == expected}
                        for key, expected in OBSERVED_COUNTS.items()},
    }

In [ ]:
clinical_report = private_call(audit_clinical_manifest, images, linkage_attestation=linkage)
print(json.dumps(clinical_report, indent=2, sort_keys=True, allow_nan=False))
print(json.dumps(compare_observed(clinical_report), indent=2, sort_keys=True, allow_nan=False))

## 4. Group/CDR counts at subject, visit and image levels

These are original clinical group/score distributions, **not model labels**.
Multiple images do not multiply visit counts. A subject may occur in more than
one Group/CDR cell across visits, so subject counts in the table are not additive.
A visit with conflicting clinical values is excluded from visit/subject table
cells rather than choosing its first image; its image records remain counted.
Unexpected category strings are reduced to fixed warning categories, never echoed.

In [ ]:
def clinical_distribution(rows):
    visits = defaultdict(list)
    for row in rows:
        visits[row.visit_key].append(row)
    table = defaultdict(lambda: {"subjects": set(), "visits": set(), "images": 0})

    def category(row):
        group = row.group if row.group in GROUPS else "<unknown_group>"
        cdr = ("missing" if row.cdr is None else
               str(float(row.cdr)) if row.cdr in CDR_VALUES else "<invalid_cdr>")
        return group, cdr

    conflicting = 0
    for key, records in visits.items():
        for row in records:
            table[category(row)]["images"] += 1
        if len({(r.group, r.cdr, r.mmse, r.mmse_field_present) for r in records}) != 1:
            conflicting += 1
            continue
        cell = table[category(records[0])]
        cell["subjects"].add(key[0])
        cell["visits"].add(key)
    return {
        "totals": {"subjects": len({r.subject_id for r in rows}),
                   "visits": len(visits), "images": len(rows)},
        "conflicting_visits_excluded_from_table": conflicting,
        "subject_memberships_are_nonadditive": True,
        "group_cdr": [{"group": group, "cdr": cdr,
                       "subjects": len(v["subjects"]), "visits": len(v["visits"]),
                       "images": v["images"]} for (group, cdr), v in sorted(table.items())],
    }

In [ ]:
print(json.dumps(private_call(clinical_distribution, images), indent=2, allow_nan=False))

## 5. Bounded, read-only Analyze `.hdr` / `.img` sample

Set `SAMPLE_SIZE` to a small positive integer (maximum 20). Only manifest-listed
pairs are considered; no recursive Drive scan occurs. Dotfiles, AppleDouble
`._*` files and `__MACOSX` paths are excluded. `.hdr` and `.img` rows for the same
pair are deduplicated. Unsafe paths or pairs linked to different visits are
flagged. Paths must be relative to the configured raw root; there is no guessed
prefix stripping or fallback search.

NiBabel reads native arrays without squeezing, reorienting, resampling, cropping,
masking, normalizing or saving. The existing `NIfTIValidator` header, dimensional
and spacing checks work through `nib.load` for Analyze too; extra checks require
**exactly 3D**, bounded dimensions/voxel count/payload, finite real-valued voxels
and a nonconstant volume. Geometry includes affine finiteness/invertibility,
plausible spacing and array axis codes. Non-RAS native axes are reported, not
changed. No NIfTI-only qform/sform requirement is imposed on Analyze files. Optional `.mat`
sidecars require separate review and are not read by this paired-file audit.

**Analyze does not reliably encode scanner/world orientation or spatial units.**
A plausible NiBabel affine/axis code does not establish anatomical left/right,
registration or millimeter units. These remain review flags, as do native-T1
provenance, motion/artifacts and visual brain coverage. Computational success is
not clinical QC approval. No file becomes `quality_flag=ok` and unsampled images
remain unvalidated. The sample is not a random-cohort or independent calibration
evaluation. Pair hashes are checked before/after reading and kept in memory only.

In [ ]:
def validate_analyze_pair(header_path, root, *, max_voxels=32_000_000,
                          max_pair_bytes=256 * 1024 * 1024):
    root = Path(root).resolve()
    header = Path(header_path)
    # Revalidate each companion after resolution, including symlinks.
    header = private_path(root, str(header.relative_to(root)))
    payload = private_path(root, str(header.with_suffix(".img").relative_to(root)))
    flags = ["analyze_world_geometry_and_units_unverified", "native_t1_not_verified",
             "visual_qc_not_performed"]
    failures = []
    result = {"computational_checks_passed": False, "quality_status": "review_required",
              "failures": failures, "quality_flags": flags}
    if not header.is_file() or not payload.is_file():
        failures.append("missing_analyze_companion")
        return result
    if header.stat().st_size > 1024 * 1024 or payload.stat().st_size > max_pair_bytes:
        failures.append("pair_exceeds_byte_limit")
        return result
    if header.with_suffix(".mat").exists():
        failures.append("analyze_sidecar_requires_review")
        return result
    before = {path: digest_file(path) for path in (header, payload)}
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            image = nib.load(str(header), mmap="r")
            if not isinstance(image, nib.analyze.AnalyzeImage) or isinstance(
                image, (nib.Nifti1Pair, nib.Nifti2Pair)
            ):
                failures.append("not_analyze_pair")
            shape = tuple(int(d) for d in image.shape)
            result["shape"] = list(shape)
            if len(shape) != 3:
                failures.append("image_not_exactly_3d")
            if any(d < 10 or d > 1024 for d in shape):
                failures.append("invalid_spatial_dimensions")
            if math.prod(shape) > max_voxels:
                failures.append("image_exceeds_voxel_limit")
            if failures:
                return result
            dtype = image.get_data_dtype()
            if not (np.issubdtype(dtype, np.integer) or np.issubdtype(dtype, np.floating)):
                failures.append("voxel_dtype_not_real_numeric")
                return result
            expected_bytes = math.prod(shape) * dtype.itemsize + image.dataobj.offset
            if payload.stat().st_size < expected_bytes:
                failures.append("truncated_analyze_payload")
                return result
            validator = NIfTIValidator()
            for report in (validator.validate_header(header),
                           validator.validate_dimensions(header),
                           validator.validate_voxel_spacing(header),
                           validator.validate_orientation(header)):
                # Never display raw validator messages/details, which may contain paths.
                failures.extend(r.check_name for r in report.errors)
                flags.extend(r.check_name for r in report.warnings)
                failures.extend(r.check_name for r in report.warnings
                                if r.check_name == "header.voxel_size")
            affine = image.affine
            if np.isfinite(affine).all() and abs(np.linalg.det(affine[:3, :3])) > 1e-10:
                result["array_axis_codes"] = list(nib.aff2axcodes(affine))
            spacing = image.header.get_zooms()[:3]
            result["header_spacing_units_unverified"] = [
                float(v) if np.isfinite(v) else None for v in spacing
            ]
            data = np.asarray(image.dataobj, dtype=np.float32)
            if not np.isfinite(data).all():
                failures.append("nonfinite_voxels")
            elif np.max(data) == np.min(data):
                failures.append("constant_volume")
    except Exception:
        failures.append("analyze_read_or_validation_error")
    finally:
        result["original_pair_unchanged"] = inputs_unchanged(before)
        if not result["original_pair_unchanged"]:
            failures.append("original_pair_changed_during_read")
        result["failures"] = sorted(set(failures))
        result["quality_flags"] = sorted(set(flags))
        result["computational_checks_passed"] = not failures
    return result


def analyze_sample(rows, root, *, sample_size=3, seed=42, max_voxels=32_000_000,
                   max_pair_bytes=256 * 1024 * 1024):
    if type(sample_size) is not int or not 1 <= sample_size <= 20:
        raise ValueError("sample_size_must_be_between_1_and_20")
    if type(max_voxels) is not int or max_voxels <= 0 or max_pair_bytes <= 0:
        raise ValueError("invalid_sample_resource_limit")
    candidates = defaultdict(set)
    excluded = Counter()
    root = Path(root).resolve()
    for row in rows:
        if hidden_metadata(row.image_path):
            excluded["hidden_metadata_image_records"] += 1
            continue
        try:
            path = private_path(root, row.image_path)
            if path.suffix not in {".hdr", ".img"}:
                excluded["unsupported_extension_image_records"] += 1
                continue
            header = private_path(root, str(path.with_suffix(".hdr").relative_to(root)))
            candidates[header].add(row.visit_key)
        except (ValueError, OSError):
            excluded["unsafe_path_image_records"] += 1
    ambiguous = {p for p, keys in candidates.items() if len(keys) != 1}
    excluded["pairs_linked_to_multiple_visits"] = len(ambiguous)
    paths = sorted(set(candidates) - ambiguous)
    random.Random(seed).shuffle(paths)
    results = []
    for index, path in enumerate(paths[:sample_size]):
        try:
            outcome = private_call(validate_analyze_pair, path, root, max_voxels=max_voxels,
                                   max_pair_bytes=max_pair_bytes)
        except RuntimeError:
            outcome = {"computational_checks_passed": False, "quality_status": "review_required",
                       "failures": ["unsafe_or_unreadable_pair"], "quality_flags": []}
        results.append({"sample_number": index + 1, **outcome})
    return {"requested_pairs": sample_size, "eligible_unique_pairs": len(paths),
            "sampled_pairs": len(results), "unsampled_pairs": max(0, len(paths) - len(results)),
            "sample_request_fulfilled": len(results) == sample_size,
            "exclusions": dict(sorted(excluded.items())),
            "computational_pass_pairs": sum(r["computational_checks_passed"] for r in results),
            "cohort_quality_approved": False, "sample_results": results}

In [ ]:
image_sample = private_call(
    analyze_sample, images, raw_root, sample_size=SAMPLE_SIZE, seed=SAMPLE_SEED,
    max_voxels=MAX_SAMPLE_VOXELS, max_pair_bytes=MAX_PAIR_BYTES,
)
print(json.dumps(image_sample, indent=2, sort_keys=True, allow_nan=False))

## 6. Proposed subject-level split, in memory only

The proposal uses the existing `SplitManager.assign_splits` on a separate,
label-free `ManifestCollection`. It does not change the clinical objects or JSON.
All visits/images of a subject stay together, including Converted visits. A
single existing fixed assignment is preserved and propagated to that subject's
unassigned rows **only in the proposal**. Contradictory existing assignments,
including holdout crossings, block the proposal. Existing non-null labels require
separate review and also block this label-free planning step.

Paths referring to the same Analyze pair under different visits/subjects block
planning. Subject and visit membership are checked again after assignment, with
the existing leakage checker plus an explicit holdout check. Requested fractions
apply to previously unassigned subjects; fixed subjects can change total ratios.
The seed is reproducible for the same input and module version, not evidence that
the cohort is balanced. Subject IDs must already be reconciled; this plan cannot
detect a person exported under multiple IDs. Cohort-wide byte-duplicate detection
is still required before freezing splits; sampling cannot establish its absence.
Empty validation/test subsets are flagged. These are
**planning partitions**, not a locked test set. Review Group/CDR distributions,
site/scanner metadata, source shortcuts and sample size before freezing anything.
A single OASIS-2 cohort cannot establish cross-source generalization or independent
calibration. No task-class stratification is possible before verified labels.

In [ ]:
def planning_record(row):
    return ManifestRecord(
        subject_id=row.subject_id, case_id=json.dumps(row.visit_key), dataset_source="oasis2",
        cohort=row.group, session_id=row.mri_visit, modality="", image_path=row.image_path,
        image_dimension="", original_diagnosis=None, task_a_label=None, task_c_label=None,
        segmentation_path=None, label_provenance="missing", license_status="unknown",
        site="", scanner="", split="validation" if row.split == "val" else row.split,
    )


def propose_splits(rows, *, root, seed=42, train_fraction=0.70, validation_fraction=0.15):
    if not rows:
        raise ValueError("empty_split_input")
    root = Path(root).resolve()
    subject_splits = defaultdict(set)
    pair_visits = defaultdict(set)
    for row in rows:
        if row.declared_task_a_label is not None or row.declared_task_c_label is not None:
            raise ValueError("existing_labels_require_separate_review")
        split = "validation" if row.split == "val" else row.split
        subject_splits[row.subject_id].add(split)
        path = private_path(root, row.image_path)
        if path.suffix in {".hdr", ".img"}:
            path = private_path(root, str(path.with_suffix(".hdr").relative_to(root)))
        pair_visits[str(path)].add(row.visit_key)
    if any(len(splits - {"unassigned"}) > 1 for splits in subject_splits.values()):
        raise ValueError("conflicting_existing_subject_splits")
    if any(len(visits) > 1 for visits in pair_visits.values()):
        raise ValueError("image_pair_linked_to_multiple_visits")
    records = []
    for row in rows:
        record = planning_record(row)
        fixed = subject_splits[row.subject_id] - {"unassigned"}
        if fixed:
            record.split = next(iter(fixed))
        records.append(record)
    manager = SplitManager(seed=seed, train_frac=train_fraction, val_frac=validation_fraction)
    proposal = manager.assign_splits(ManifestCollection(records))
    leakage = manager.check_leakage(proposal)
    after_subject = defaultdict(set)
    after_visit = defaultdict(set)
    for record in proposal.records:
        after_subject[record.subject_id].add(record.split)
        after_visit[(record.subject_id, record.session_id)].add(record.split)
    if not leakage.is_clean or any(len(s) != 1 for s in after_subject.values()) or any(
        len(s) != 1 for s in after_visit.values()
    ):
        raise ValueError("proposal_failed_subject_or_visit_leakage_checks")
    summaries = {}
    for split in ("train", "validation", "test", "holdout"):
        subset = [r for r in proposal.records if r.split == split]
        summaries[split] = {"subjects": len({r.subject_id for r in subset}),
                            "visits": len({r.case_id for r in subset}), "images": len(subset)}
    return proposal, {
        "scope": "in-memory planning only; not final or clinically stratified",
        "seed": seed, "requested_train_fraction": train_fraction,
        "requested_validation_fraction": validation_fraction,
        "existing_assignments_preserved": True, "subject_and_visit_leakage_checks_passed": True,
        "all_ranked_splits_nonempty": all(summaries[s]["subjects"] > 0
                                         for s in ("train", "validation", "test")),
        "counts": summaries,
    }


def summarize_readiness(proposal, root):
    # Metadata-only; never declare OASIS-2 approved or pass Analyze through NIfTI-only QC.
    readiness = audit_readiness(proposal, {}, image_root=Path(root), check_images=False)
    return {"ready": readiness["ready"], "assessment": readiness["assessment"],
            "eligibility_scope": readiness["eligibility_scope"],
            "counts": readiness["counts"], "blockers": readiness["blockers"]}

In [ ]:
# Sensitive proposal records stay in memory. Never display or serialize `proposal`.
proposal, split_summary = private_call(
    propose_splits, images, root=raw_root, seed=SPLIT_SEED, train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
)
print(json.dumps(split_summary, indent=2, allow_nan=False))
for split in ("train", "validation", "test", "holdout"):
    # Use original clinical rows; the split plan supplies membership only.
    members = {r.subject_id for r in proposal.records if r.split == split}
    distribution = private_call(
        clinical_distribution, [r for r in images if r.subject_id in members],
    )
    print(json.dumps({"proposed_split": split, **distribution}, indent=2, allow_nan=False))
print(json.dumps(private_call(summarize_readiness, proposal, raw_root), indent=2, allow_nan=False))

## 7. Final preservation and interpretation

The final cell rechecks metadata bytes and reports software provenance and audit
scope. The notebook has no manifest/split/report writer and never saves images.
If a preservation check fails, stop and investigate concurrent changes privately.
The pair hashes establish unchanged sampled files during each read, not immutable
storage or correctness of the original join. A separate authorized review still
needs to resolve the two Nondemented/CDR 0.5 visits, verify diagnosis/MRI-normality
evidence and source access, inspect native image orientation and QC, and approve
any future split or label sidecar. Original masks/absence of segmentation and
missing diagnoses must never be used to infer normality.

An independent, patient-disjoint and suitably source/site-aware evaluation cohort
and separate calibration evaluation remain necessary. Uncertainty alone cannot
reliably detect unseen diseases. Nothing here validates Task C's performance or
changes its output contract. Keep raw datasets isolated; a future virtual merged
manifest does not authorize pooling or copying images.

In [ ]:
metadata_preserved = private_call(inputs_unchanged, input_fingerprints)
if not metadata_preserved:
    raise RuntimeError("Input preservation check failed; investigate privately before continuing.")
print(json.dumps({
    "module_baseline_commit": MODULE_COMMIT,
    "python_version": sys.version.split()[0],
    "dependencies": {name: version(name) for name in ("numpy", "nibabel")},
    "policy_version": clinical_report["policy_version"],
    "sample_seed": SAMPLE_SEED, "sample_size": SAMPLE_SIZE, "split_seed": SPLIT_SEED,
    "metadata_bytes_unchanged": metadata_preserved,
    "sampled_pair_preservation_failures": sum(
        not r.get("original_pair_unchanged", False) for r in image_sample["sample_results"]
    ),
    "source_manifest_or_split_files_written": False,
    "final_diagnoses_confirmed": False, "final_task_labels_assigned": False,
    "task_a_or_c_release_ready": False,
}, indent=2, allow_nan=False))